# Session 2 · Day 1 - Maps of the TESDA Institutions

The coordinate audit left a geocoded institution table with a quality flag on every pin. That table becomes three kinds of map here: a **point map** that shows every center, a **density heat map** that turns thousands of overlapping dots into a readable surface, and an **interactive map** for panning and clicking. Each view answers a different question about *where* TESDA capacity sits, and each has a format that survives on GitHub and one that needs a live notebook.

Data streams from the public bucket `https://storage.googleapis.com/tesda-datasets/data/`. The audit itself is the subject of `S2_D1_03_institutions_and_coordinate_audit`; polygon choropleths over admin boundaries are the subject of `S2_D1_02_admin_boundaries`.

**How to use this notebook**

- ▶ run the cells in order; each map builds on the table loaded at the top.
- ✏️ fill the code between `### START CODE HERE ###` and `### END CODE HERE ###`.
- ✅ run the check cell right after an exercise to confirm the answer.
- ❓ answer the short question with a computed value.
- 🐞 run the flawed cell, read what breaks, then apply the fix.
- 🎛️ drag the slider to feel how one parameter changes the map.

In [ ]:
# Run once per session (skip if already installed).
!pip -q install pandas geopandas folium matplotlib mapclassify contextily

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
STORE_CRS = 4326     # WGS 84, degrees of longitude and latitude, the storage and web-map system
WEB_CRS = 3857       # Web Mercator, meters, what tile basemaps and web maps draw in
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 55)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def add_basemap(ax, crs=WEB_CRS):
    """Draw a light tile basemap under the axes, and skip quietly when offline."""
    try:
        import contextily as cx
        cx.add_basemap(ax, crs=crs, source=cx.providers.CartoDB.Positron, attribution_size=6)
    except Exception as e:
        print("(basemap skipped:", type(e).__name__, ")")

## Load the geocoded institutions

One row is one TESDA institution. The audit wrote a `coord_status` on each: `valid` for a pin inside the Philippines, `out_of_bounds` for a coordinate that lands off the map, `null_coords` for a missing pin. Mapping starts from that flag, so load the table and read the counts first.

In [ ]:
coords = pd.read_parquet(BASE + "tesda_coordinates.parquet")
print(f"{len(coords):,} institutions x {coords.shape[1]} columns")
print("coordinate status:", coords["coord_status"].value_counts().to_dict())
coords[["name", "region", "city_municipality", "latitude", "longitude", "coord_status"]].head()

**Expected output:** about 8,000 institutions, with `coord_status` roughly `valid` 7,891, `out_of_bounds` 108, `null_coords` 8.

## Keep the pins the audit trusts

A map reads every row as a location, so an `out_of_bounds` pin does real damage. One latitude in the billions stretches the axes until the whole country collapses to a dot. Build a `GeoDataFrame` from the `valid` rows alone, in `EPSG:4326`. The flawed cell near the end shows what the unfiltered version looks like.

In [ ]:
located = coords[coords["coord_status"] == "valid"].copy()
pts = gpd.GeoDataFrame(
    located,
    geometry=gpd.points_from_xy(located["longitude"], located["latitude"]),
    crs=STORE_CRS,
)
print(f"{len(pts):,} mapped institutions")
print("longitude span:", round(pts.geometry.x.min(), 2), "to", round(pts.geometry.x.max(), 2))
print("latitude span: ", round(pts.geometry.y.min(), 2), "to", round(pts.geometry.y.max(), 2))

**Expected output:** about 7,891 mapped institutions, longitude roughly 117 to 126 and latitude roughly 4.8 to 20.5, the real extent of the Philippines.

## A point map of every center

The plainest spatial view puts one dot per institution. A static `matplotlib` figure renders straight to the page and survives on GitHub, where an interactive widget does not. Project to Web Mercator first so the dots line up with the tile basemap, then draw them small and semi-transparent so dense areas darken on their own.

In [ ]:
ax = pts.to_crs(WEB_CRS).plot(figsize=(7, 9), markersize=4, color="#d1495b", alpha=0.4)
add_basemap(ax)
ax.set_axis_off()
ax.set_title(f"{len(pts):,} TESDA institutions with a usable pin")
plt.show()

**Expected output:** the national outline picked out in red dots, dense around Metro Manila, Central Luzon, Cebu, and Davao, sparse across the uplands and the far south.

## Color the points by region

The same dots carry a category when colored by `region`. This is the quickest read on which parts of the country the 17 regions cover and where their institutions cluster. A categorical colormap assigns one hue per region.

In [ ]:
ax = (pts.to_crs(WEB_CRS)
         .plot(figsize=(7, 9), column="region", categorical=True, markersize=5,
               legend=True, legend_kwds={"fontsize": 6, "loc": "upper left", "ncol": 2},
               cmap="tab20", alpha=0.7))
add_basemap(ax)
ax.set_axis_off()
ax.set_title("Institutions by region")
plt.show()

## Density as a heat map

Thousands of dots overplot: once pins pile on top of one another, a darker patch no longer says how much darker. A heat map fixes that by binning nearby points into a smooth surface, so intensity reads as count per area. `folium.plugins.HeatMap` takes a list of `[lat, lon]` pairs and renders over a slippy map. Interactive output needs a live notebook, so a static version follows for sharing.

In [ ]:
m = folium.Map(location=[12.8, 122.0], zoom_start=5, tiles="cartodbpositron")
HeatMap(
    data=pts[["latitude", "longitude"]].values.tolist(),
    radius=9, blur=7, min_opacity=0.3,
).add_to(m)
m

**Expected output:** a zoomable map with hot cores over Metro Manila and the Cebu and Davao corridors, cooling toward the sparsely served interior.

### A static density for sharing

`hexbin` aggregates the points into hexagonal cells and colors each by its count, the same density idea in a figure that renders on GitHub. The color scale reports the raw number of institutions per cell, which a smooth heat map hides behind its gradient.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
hb = ax.hexbin(pts.geometry.x, pts.geometry.y, gridsize=40, mincnt=1, cmap="magma")
fig.colorbar(hb, ax=ax, shrink=0.5, label="institutions per cell")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Institution density (hexagonal bins)")
plt.show()

**Expected output:** a few bright cells over Metro Manila and the central Visayas reaching into the hundreds, against a dark background of single-count cells.

## An interactive map to explore

`GeoDataFrame.explore` builds a Leaflet map where each pin carries a tooltip, so a reader can pan, zoom, and click a center to read its name and region. The method needs `folium` and `mapclassify`. The call is wrapped so it degrades to the static point map when a dependency or the network is missing.

In [ ]:
keep = ["name", "region", "city_municipality", "type_of_institution", "geometry"]
try:
    explorer = pts[keep].explore(
        column="region", categorical=True, cmap="tab20",
        tooltip=["name", "region", "city_municipality"],
        marker_kwds={"radius": 4}, style_kwds={"fillOpacity": 0.8, "weight": 0.3},
        legend=False, tiles="cartodbpositron",
    )
    display(explorer)
except Exception as e:
    print("(interactive map skipped:", type(e).__name__, "- showing the static map)")
    ax = pts.to_crs(WEB_CRS).plot(figsize=(7, 9), markersize=4, color="#d1495b", alpha=0.4)
    add_basemap(ax); ax.set_axis_off(); plt.show()

## Count the institutions in each region

A map shows the pattern; a count names it. Grouping the mapped pins by `region` turns the picture into a ranked table, the descriptive statistic behind the colors above. A sorted horizontal bar is the readable form for 17 categories.

In [ ]:
by_region = pts["region"].value_counts().sort_values()
ax = by_region.plot.barh(figsize=(7, 6), color="#30638e")
ax.set_xlabel("institutions with a usable pin")
ax.set_title("Mapped institutions by region")
for i, v in enumerate(by_region):
    ax.text(v + 5, i, f"{v:,}", va="center", fontsize=7)
plt.show()

### ✏️ Exercise - the three densest regions

Build `top3`, the three regions with the most mapped institutions, as a `Series` indexed by region name and sorted from most to fewest. Base it on `pts`, so only `valid` pins count.

**Hint:** `value_counts()` already sorts from most to fewest; take the first three.

In [ ]:
### START CODE HERE ###
top3 = None
### END CODE HERE ###
print(top3)

## ✅ Check the result

In [ ]:
ref = pts["region"].value_counts().head(3)
assert top3 is not None, "assign top3 in the exercise cell"
assert list(top3.index) == list(ref.index), "the three regions or their order do not match"
assert list(top3.values) == list(ref.values), "the counts do not match"
print("\u2705 Passed! Densest region:", top3.index[0], "with", f"{top3.iloc[0]:,}", "institutions")

### ❓ Question

How many mapped institutions sit in the single densest region? Assign the integer to `q_densest_count`.

In [ ]:
### START CODE HERE ###
q_densest_count = None      # <- the count in the top region
### END CODE HERE ###

expected = int(pts["region"].value_counts().iloc[0])
assert q_densest_count == expected, f"not yet - the densest region holds {expected} pins"
print("\u2705 Correct:", f"{expected:,}", "institutions in", pts["region"].value_counts().index[0])

## 🐞 Spot the flaw - mapping raw coordinates

A naive map trusts every `latitude` and `longitude` in the file and skips the audit flag. The next cell plots all 8,007 raw pins. Run it, read the axis range it prints, and watch the country disappear.

In [ ]:
raw = coords.dropna(subset=["latitude", "longitude"])
print("raw latitude range:", raw["latitude"].min(), "to", raw["latitude"].max())
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(raw["longitude"], raw["latitude"], s=4, color="#d1495b", alpha=0.4)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Every raw pin, no quality flag")
plt.show()

A handful of `out_of_bounds` pins, one latitude above ten billion, blow the axes to a scale where all 7,891 real pins collapse into a single point near the origin. The fix is the `coord_status` filter already applied to `pts`: map the pins the audit marked `valid` and leave the rest out, or repair them first. The same rule protects the heat map, since one stray pin would drag a hot spot into the ocean.

In [ ]:
ax = pts.plot(figsize=(6, 6), markersize=4, color="#2d6a4f", alpha=0.4)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Valid pins only, the country back in view")
plt.show()

## 🎛️ Play around - the heat map radius

A heat map has no single correct look. The `radius` sets how far each point spreads, so a small radius shows tight local clusters and a large one smooths them into broad zones. Drag the slider and watch the hot spots merge.

In [ ]:
def heat(radius=9):
    m = folium.Map(location=[12.8, 122.0], zoom_start=5, tiles="cartodbpositron")
    HeatMap(pts[["latitude", "longitude"]].values.tolist(),
            radius=radius, blur=max(3, radius - 2), min_opacity=0.3).add_to(m)
    return m

try:
    from ipywidgets import interact, IntSlider
    interact(heat, radius=IntSlider(min=4, max=24, step=2, value=9))
except Exception:
    display(heat(9))   # static fallback outside Jupyter

## Recap

- A static point map renders on GitHub and shows every center; an interactive `.explore` map adds pan, zoom, and tooltips in a live notebook.
- Overlapping dots hide quantity. A heat map and a `hexbin` grid both recover density; the heat map reads as a smooth gradient, the hexbin reports the count per cell.
- Coloring and counting by `region` turn the same pins into a descriptive summary of where capacity sits.
- Every map depends on the `coord_status` flag. An unaudited pin does not just add noise, it rescales the whole picture.

## References & further reading

- GeoPandas mapping and `explore`: https://geopandas.org/en/stable/docs/user_guide/interactive_mapping.html
- Matplotlib `hexbin`: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hexbin.html
- Folium `HeatMap` plugin: https://python-visualization.github.io/folium/latest/user_guide/plugins/heatmap.html
- contextily basemaps: https://contextily.readthedocs.io/
- Companion notebooks: `S2_D1_02_admin_boundaries` (polygon choropleths), `S2_D1_03_institutions_and_coordinate_audit` (the quality flag used here).